# Week 6 Capstone — End-to-End Secondary-Structure Prediction Pipeline

CSOT × iGEM × ARIES 2026

This notebook loads the saved Week 4 BiLSTM+ESM2 checkpoint (no retraining),
runs it on a self-constructed test set, and reproduces every number in
`report.pdf`: `predictions.csv`, `predictions_smoothed.csv`, the error
analysis, and the biological-inference results.

**Required upload before running (Runtime → Run all):**
- `bilstm_esm2.pt`
- `bilstm_esm2_config.json`
- `test.fasta`
- `test_labels.csv` (ground-truth DSSP labels — used only for scoring, never fed to the model)


## 1. Setup & seeds

In [ ]:
!pip install fair-esm biopython -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seeds set to {SEED}. Device: {device}")

In [ ]:
# Upload the four required files if not already present in the Colab filesystem.
import os
from google.colab import files

required = ["bilstm_esm2.pt", "bilstm_esm2_config.json", "test.fasta", "test_labels.csv"]
missing = [f for f in required if not os.path.exists(f)]

if missing:
    print(f"Missing files: {missing}. Please upload them now.")
    uploaded = files.upload()
else:
    print("All required files already present.")

## 2. Load artefacts

No training happens in this notebook. The model architecture is rebuilt from
the saved config, then the saved weights are loaded directly into it.

In [ ]:
import json
import torch.nn as nn

MODEL_PATH = "bilstm_esm2.pt"
CONFIG_PATH = "bilstm_esm2_config.json"

with open(CONFIG_PATH) as f:
    config = json.load(f)
print("Loaded config:", config)

class BiLSTM_ESM(nn.Module):
    """Identical architecture to the one trained in Week 4."""
    def __init__(self, esm_dim=320, hidden=64, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(esm_dim, hidden, num_layers=layers,
                             batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Linear(2 * hidden, 3)

    def forward(self, x):          # x: (B, L, esm_dim)
        h, _ = self.lstm(x)
        return self.head(h)

model = BiLSTM_ESM(**config)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f"Loaded weights from {MODEL_PATH}")
print(model)

## 3. The `SecStructPredictor` class

Wraps ESM2 feature extraction (Stage 3), the loaded model (Stage 4), and
argmax + optional smoothing (Stage 5) into one callable, per the Week 6
pipeline blueprint. ESM2 itself is frozen and pretrained — it is loaded here,
not trained.

In [ ]:
import esm

esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
esm_model = esm_model.to(device)
esm_model.eval()
for p in esm_model.parameters():
    p.requires_grad = False

batch_converter = alphabet.get_batch_converter()
ESM_DIM = 320

def get_esm_embedding(seq):
    """Stage 3: sequence -> (L, 320) frozen ESM2 embedding."""
    data = [("protein", seq)]
    _, _, tokens = batch_converter(data)
    tokens = tokens.to(device)
    with torch.no_grad():
        out = esm_model(tokens, repr_layers=[6])
    reps = out["representations"][6][0]
    return reps[1:-1].cpu()   # (L, 320), strip BOS/EOS tokens

print("ESM2 (esm2_t6_8M_UR50D) loaded and frozen.")

In [ ]:
Q3_LABELS = {0: "H", 1: "E", 2: "C"}

class SecStructPredictor:
    """Wraps a trained model and all preprocessing into a single callable."""

    def __init__(self, model, feature_fn, label_map=Q3_LABELS):
        self.model = model
        self.feature_fn = feature_fn
        self.label_map = label_map

    def predict(self, sequence):
        X = self.feature_fn(sequence)                    # (L, 320)
        with torch.no_grad():
            t = X.unsqueeze(0).float().to(device)          # (1, L, 320)
            logits = self.model(t)                          # (1, L, 3)
            indices = logits.argmax(-1)[0].cpu().numpy()
        return "".join(self.label_map[i] for i in indices)

    def predict_from_fasta(self, fasta_path):
        """Returns a list of (protein_id, sequence, prediction_string) tuples."""
        from Bio import SeqIO
        results = []
        for record in SeqIO.parse(fasta_path, "fasta"):
            seq = str(record.seq)
            pred = self.predict(seq)
            assert len(pred) == len(seq), f"length mismatch for {record.id}"
            results.append((record.id, seq, pred))
        return results

    def save_predictions(self, results, out_path):
        """Per-residue CSV matching the assignment's required schema."""
        with open(out_path, "w") as f:
            f.write("protein_id,position,residue,predicted_ss\n")
            for protein_id, seq, pred in results:
                for i, (aa, ss) in enumerate(zip(seq, pred), start=1):
                    f.write(f"{protein_id},{i},{aa},{ss}\n")


def smooth_predictions(pred_string, min_segment=3):
    """Sliding-window majority-vote smoothing (Week 6 notes)."""
    labels = list(pred_string)
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(labels):
            cls = labels[i]
            j = i
            while j < len(labels) and labels[j] == cls:
                j += 1
            if (j - i) < min_segment:
                replacement = labels[i-1] if i > 0 else (labels[j] if j < len(labels) else cls)
                for k in range(i, j):
                    labels[k] = replacement
                changed = True
            i = j
    return "".join(labels)

predictor = SecStructPredictor(model, get_esm_embedding)
print("SecStructPredictor ready.")

## 4. Run on the test set

In [ ]:
import datetime

results = predictor.predict_from_fasta("test.fasta")

# Reproducibility check: calling predict() twice on the same input gives identical output
_check = predictor.predict(results[0][1])
assert _check == results[0][2], "Non-deterministic output detected!"
print(f"Determinism check passed on {results[0][0]}.")

predictor.save_predictions(results, "predictions.csv")

smoothed_results = [(pid, seq, smooth_predictions(pred, min_segment=3)) for pid, seq, pred in results]
predictor.save_predictions(smoothed_results, "predictions_smoothed.csv")

# Stamp: which model + dataset produced this run
stamp = (f"# predictions.csv generated {datetime.datetime.now().isoformat()} "
         f"from model={MODEL_PATH} config={config} dataset=test.fasta ({len(results)} proteins)")
print(stamp)
with open("predictions_stamp.txt", "w") as f:
    f.write(stamp + "\n")

for pid, seq, pred in results:
    print(f"{pid} ({len(seq)} residues): {pred[:60]}{'...' if len(seq)>60 else ''}")

## 5. Error analysis

Scored against `test_labels.csv`, the self-constructed ground-truth DSSP
labels for the 5 test proteins. This file was never used as model input.

In [ ]:
import pandas as pd

pred_raw = pd.read_csv("predictions.csv")
pred_smooth = pd.read_csv("predictions_smoothed.csv")
truth = pd.read_csv("test_labels.csv")

def merge_and_score(pred_df, label):
    m = truth.merge(pred_df, on=["protein_id", "position"], suffixes=("_true", "_pred"))
    assert len(m) == len(truth), f"row count mismatch: {len(m)} vs {len(truth)}"
    acc = (m["true_ss"] == m["predicted_ss"]).mean()
    print(f"Overall Q3 accuracy ({label}): {acc:.4f}  ({len(m)} residues)")
    return m

m_raw = merge_and_score(pred_raw, "raw")
m_smooth = merge_and_score(pred_smooth, "smoothed")

In [ ]:
labels = ["H", "E", "C"]
cm = pd.crosstab(m_raw["true_ss"], m_raw["predicted_ss"]).reindex(index=labels, columns=labels, fill_value=0)
print("Confusion matrix (raw), rows=true, cols=pred:")
print(cm)
print()
for l in labels:
    row = cm.loc[l]
    total = row.sum()
    print(f"True {l}: n={total}, recall={row[l]/total:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix (raw)")
for i in range(3):
    for j in range(3):
        v = cm.values[i, j]
        ax.text(j, i, str(v), ha="center", va="center",
                color="white" if v > cm.values.max()/2 else "black")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
def label_boundaries(ss_string):
    n = len(ss_string)
    boundary = [False] * n
    for i in range(n):
        left_diff = (i == 0) or (ss_string[i] != ss_string[i-1])
        right_diff = (i == n-1) or (ss_string[i] != ss_string[i+1])
        boundary[i] = left_diff or right_diff
    return boundary

m_raw = m_raw.sort_values(["protein_id", "position"]).reset_index(drop=True)
bflags = []
for pid, group in m_raw.groupby("protein_id", sort=False):
    true_str = "".join(group["true_ss"].tolist())
    bflags.extend(label_boundaries(true_str))
m_raw["is_boundary"] = bflags

interior = m_raw[~m_raw["is_boundary"]]
boundary = m_raw[m_raw["is_boundary"]]
q3_interior = (interior["true_ss"] == interior["predicted_ss"]).mean()
q3_boundary = (boundary["true_ss"] == boundary["predicted_ss"]).mean()
print(f"Interior Q3: {q3_interior:.3f} (n={len(interior)})   Boundary Q3: {q3_boundary:.3f} (n={len(boundary)})")
print(f"Gap: {q3_interior - q3_boundary:.3f}")

In [ ]:
def run_lengths_and_accuracy(true_labels, pred_labels):
    i = 0
    out = []
    while i < len(true_labels):
        cls = true_labels[i]; j = i
        while j < len(true_labels) and true_labels[j] == cls:
            j += 1
        run_len = j - i
        correct = sum(pred_labels[k] == cls for k in range(i, j)) / run_len
        out.append((cls, run_len, correct))
        i = j
    return out

all_runs = []
for pid, group in m_raw.groupby("protein_id", sort=False):
    t = "".join(group["true_ss"].tolist())
    p = "".join(group["predicted_ss"].tolist())
    all_runs.extend(run_lengths_and_accuracy(t, p))

runs_df = pd.DataFrame(all_runs, columns=["cls", "run_len", "acc"])
bins = [0, 3, 6, 10, 20, 1000]
bucket_labels = ["1-3", "4-6", "7-10", "11-20", "20+"]
runs_df["bucket"] = pd.cut(runs_df["run_len"], bins=bins, labels=bucket_labels)

print("Segment-length effect:")
print(runs_df.groupby("bucket", observed=True).agg(n_segments=("acc", "size"), mean_acc=("acc", "mean")))
print()
print("By class:")
print(runs_df.groupby("cls").agg(n_segments=("acc", "size"), mean_run_len=("run_len", "mean"), mean_acc=("acc", "mean")))

In [ ]:
per_protein = m_raw.groupby("protein_id").apply(lambda g: (g["true_ss"] == g["predicted_ss"]).mean())
print("Per-protein Q3 (hardest first):")
print(per_protein.sort_values())

fig, ax = plt.subplots(figsize=(5, 3.5))
per_protein_sorted = per_protein.sort_values()
ax.barh(per_protein_sorted.index, per_protein_sorted.values,
        color=["#C44E52" if v < 0.8 else "#4C72B0" for v in per_protein_sorted.values])
ax.set_xlabel("Q3 accuracy"); ax.set_xlim(0, 1.0)
ax.set_title("Per-Protein Q3 Accuracy")
plt.tight_layout()
plt.savefig("per_protein_q3.png", dpi=150)
plt.show()

## 6. Biological inference

Applied to all 5 test proteins (the assignment requires at least 3 named
proteins).

In [ ]:
def ss_composition(pred_string):
    n = len(pred_string)
    return {"H%": 100*pred_string.count("H")/n, "E%": 100*pred_string.count("E")/n, "C%": 100*pred_string.count("C")/n}

def find_long_runs(pred_string, label, min_len):
    runs = []; i = 0
    while i < len(pred_string):
        if pred_string[i] == label:
            j = i
            while j < len(pred_string) and pred_string[j] == label:
                j += 1
            if j - i >= min_len:
                runs.append((i+1, j))
            i = j
        else:
            i += 1
    return runs

for pid, seq, pred in results:
    comp = ss_composition(pred)
    tm = find_long_runs(pred, "H", min_len=20)
    idr = find_long_runs(pred, "C", min_len=30)
    print(f"{pid} (n={len(pred)})")
    print(f"  Composition: H={comp['H%']:.1f}%  E={comp['E%']:.1f}%  C={comp['C%']:.1f}%")
    print(f"  TM-helix candidates (>=20 consec H): {tm}")
    print(f"  IDR candidates (>=30 consec C): {idr}")
    print()

### Cross-check: trypsin catalytic triad (1TGTA)

Trypsin's catalytic His and Ser sit in two sequence motifs conserved across
the whole chymotrypsin-like serine protease family: the catalytic histidine
within an "AAHC" motif, and the catalytic serine within the "GDSGGP"
nucleophile-elbow motif. Both are located directly in the downloaded
sequence and compared against DSSP ground truth.

In [ ]:
import re

g = m_raw[m_raw.protein_id == "1TGTA"].sort_values("position")
seq = "".join(g["residue"])
pred_str = "".join(g["predicted_ss"])
true_str = "".join(g["true_ss"])

his_motif = re.search("AAHC", seq)
ser_motif = re.search("GDSGGP", seq)
his_pos = his_motif.start() + 3
ser_pos = ser_motif.start() + 3

for name, pos in [("Catalytic His", his_pos), ("Catalytic Ser", ser_pos)]:
    print(f"{name} (residue {seq[pos-1]}{pos}): DSSP truth={true_str[pos-1]}  model pred={pred_str[pos-1]}  "
          f"{'MATCH' if true_str[pos-1]==pred_str[pos-1] else 'MISMATCH'}")

## Reproducibility checklist

In [ ]:
checklist = {
    "Model weights saved with the hyperparameter config that produced them": True,
    "Vectoriser/scaler fitted only on training chains, saved and loaded at test time": "N/A (ESM2 is frozen, pretrained; no fitted scaler used)",
    "Random seed set (numpy, torch, python random) before any split or inference": True,
    "The train/val/test split is saved or deterministically reproducible from the seed": True,
    "Running the pipeline twice on the same input produces identical output": True,
    "predictions.csv is stamped with which model file and dataset produced it": True,
}

print("Reproducibility checklist:")
for item, status in checklist.items():
    mark = "[x]" if status is True else f"[~] ({status})"
    print(f"{mark} {item}")